# Datamine MAKEDTM Process Example

This notebook demonstrates how to configure and run the **`makedtm`** process wrapper in `dmstudio`.

## Process Description

## MAKEDTM

**MAKEDTM** generates a DTM file from points data in a file, which is optionally subdivided or constrained using one of the following inputs:

* Closed perimeter data
* String files

Point coordinate fields will be automatically detected if possible, but can be reviewed/edited using the Fields tab.

* **Note** (For interactive DTM creation from loaded string or point data, see [dtm-create ("md")](<../command_help/dtm-create.md>).):

### Make Diagonals Consistent

You can form a DTM from one or more input string or points objects (or a combination). However, where point data is common between multiple objects (meaning there are coincident data points in the set used to form the DTM), it is possible for surface triangulation to be performed in a different way, and often unexpectedly so, compared to the outcome if data were sourced from a single object (without coincident points).

To mitigate this possibility, the Make Diagonals Consistent option can be checked to ensure the surface generated between disparate data sets is performed identically, ensuring the combined DTM matches the triangulated output from a single data object input.

* **Note** (Selecting this option can introduce a performance hit, so where large coincident data overlaps are known to occur between input objects, it may be more efficient to combine data first into a single object (say, using the [Copy from Object(s)](<../COMMON/CopyDataFromDialog.md>) screen) before generating a DTM.):

### Input Files:

* **perimin** (Strings):
  Input perimeter file containing XP,YP,ZP,PTN, PVALUE fields. Closed strings will be
  used as boundaries to the triangulation, and may be included in the triangulation if
  @INCPERIM is 1.
  Required=No

* **stringin** (Strings):
  Input string file containing XP,YP,ZP,PTN and PVALUE fields. String segments are
  included in the triangulation as 3D edge constraints, breaklines. Strings may be open or
  closed.
  Required=No

* **pointsin** (Points):
  Input point file containing XPT,YPT,ZPT fields.
  Required=No

### Output Files:

* **wiretr** (Wireframe triangles):
  Output wireframe triangle file.
  Required=Yes

* **wirept** (Wireframe points):
  Output wireframe points file.
  Required=Yes

### Fields:

* **xpt** (Numeric : **PERIMIN** , **STRINGIN** or **POINTSIN**):
  X field in input file
  Default=Undefined
  Required=Yes

* **ypt** (Numeric : **PERIMIN** , **STRINGIN** or **POINTSIN**):
  Y field in input file
  Default=Undefined
  Required=Yes

* **zpt** (Numeric : **PERIMIN** , **STRINGIN** or **POINTSIN**):
  Z field in input file
  Default=Undefined
  Required=Yes

### Parameters:

* **boundary**:
  Boundary specifier for perimeters:
  Range=
  Values=
  Default=
  Required=No

* **incperim**:
  Include perimeter strings in the triangulation :
  Range=
  Values=
  Default=
  Required=No

* **tol**:
  Tolerance distance below which points can be moved to avoid sliver triangles around
  breaklines
  Range=Undefined
  Values=Undefined
  Default=0.00001
  Required=No

* **flattri**:
  Avoid flat triangles:
  Range=
  Values=
  Default=
  Required=No

* **trim**:
  Trim edge triangles:
  Range=
  Values=
  Default=
  Required=No

* **trimang**:
  Minimum (2D) angle allowed in an edge triangle to avoid trimming
  Range=0,360
  Values=Undefined
  Default=0
  Required=No

* **trimlen**:
  Maximum (2D) edge length allowed in an edge triangle to avoid trimming
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **crest**:
  Add automatic crest spurs to minimise upper plateaus:
  Range=
  Values=
  Default=
  Required=No

* **cresadj**:
  Vertical distance to offset the crest spurs
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **valley**:
  Add automatic valley spurs to minimise lower plateaus:
  Range=
  Values=
  Default=
  Required=No

* **valleyadj**:
  Vertical distance to offset the valley spurs
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **key**:
  Add automatic key spurs:
  Range=
  Values=
  Default=
  Required=No

* **diagonal**:
  Manage triangulation between multiple input data objects.
  Range=
  Values=
  Default=
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('makedtm')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute makedtm
print("Running makedtm...")
dm_cmd.makedtm(
    stringin_i='t_SurfaceTriangles',  # required
    pointsin_i='t_SurfacePointsPt',  # required
    wiretr_o='t_makedtm_out',  # required
    wirept_o='t_makedtm_out',  # required
    xpt_f='optional',  # required
    ypt_f='optional',  # required
    zpt_f='optional',  # required
    # perimin_i='optional',  # optional
    # boundary_p='optional',  # optional
    # incperim_p='optional',  # optional
    # tol_p=1e-05,  # optional
    # flattri_p='optional',  # optional
    # trim_p='optional',  # optional
    # trimang_p=0,  # optional
    # trimlen_p=0,  # optional
    # crest_p='optional',  # optional
    # cresadj_p=0,  # optional
    # valley_p='optional',  # optional
    # valleyadj_p=0,  # optional
    # key_p='optional',  # optional
    # diagonal_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("makedtm execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_makedtm_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")